Если вызывать `execute()` в цикле, каждый вызов отправляет отдельный запрос к базе данных. При сотнях или тысячах записей это ощутимо замедляет работу.

Метод `executemany()` решает эту проблему. Он принимает запрос и список значений, а `psycopg` объединяет всё в один пакет и отправляет за один раз:

```python
users = [
    ("Bob", "bob@mail.com"),
    ("Alice", "alice@mail.com"),
    ("John", "john@mail.com"),
]
curs.executemany("INSERT INTO users (name, email) VALUES (%s, %s)", users)
```

Путаница из‑за одинакового имени `users` — чисто «человеческий» момент: в коде имя переменной в Python случайно совпало с именем таблицы в SQL. В реальной практике это встречается, но опытные разработчики стараются так не делать — чтобы код был понятнее.

`Как делают на практике, чтобы не путаться`

Самый простой и распространённый приём — давать переменной в `Python` имя во множественном числе или с суффиксом, а в `SQL` оставлять как есть (потому что имена таблиц обычно и так во множественном):

```python
users_data = [
    ("Bob", "bob@mail.com"),
    ("Alice", "alice@mail.com"),
    ("John", "john@mail.com"),
]

curs.executemany(
    "INSERT INTO users (name, email) VALUES (%s, %s)",
    users_data
)
```

Или вариант с префиксом `batch_`, `rows_`, `values_` — это сразу даёт понять, что это пакет данных, а не таблица:

```python

batch_users = [...]
curs.executemany("INSERT INTO users (...) VALUES (...)", batch_users)
```

Ещё один популярный стиль — называть по смыслу: `new_users`, `incoming_users`, `users_to_insert`.

`Пример более реалистичного кода`

```python

import psycopg

new_users = [
    ("Bob", "bob@mail.com"),
    ("Alice", "alice@mail.com"),
    ("John", "john@mail.com"),
]

with psycopg.connect("dbname=hexlet_test user=postgres password=your_password") as conn:
    with conn.cursor() as cur:
        cur.executemany(
            "INSERT INTO users (name, email) VALUES (%s, %s)",
            new_users
        )
        conn.commit()
```

Здесь `users` — таблица в БД, new_users` — список данных в Python. Никакой двусмысленности.